# Data Cleaning

## Business Objective

The objective of this notebook is to transform the raw e-commerce datasets into reliable, consistent, and analysis-ready datasets while preserving the integrity of the original raw data.

The cleaning process covers six tables:

- `users`
- `products`
- `orders`
- `order_items`
- `reviews`
- `events`

All cleaning operations are performed on working copies of the raw datasets. Each transformation is documented and validated to ensure transparency, reproducibility, and readiness for downstream analysis.

## Cleaning Methodology

The cleaning workflow follows a structured process:

1. Load raw datasets  
2. Create independent working copies  
3. Standardize datatypes  
4. Handle invalid values  
5. Remove exact duplicate rows  
6. Resolve duplicate primary keys where appropriate  
7. Standardize categorical/text fields  
8. Validate cleaned datasets  
9. Export cleaned datasets  

The goal is not to make the data artificially perfect, but to make it reliable and suitable for business analysis while minimizing information loss.

## Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

## Set Project Directory

In [ ]:
os.chdir("/Users/saadmaher/Desktop/Data Science/Project portfolio/ecommerce-data-analysis")

PROJECT_ROOT = Path(os.getcwd())
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
CLEAN_DATA_DIR = PROJECT_ROOT / "data" / "cleaned"

CLEAN_DATA_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT

## Load Raw Datasets

In [ ]:
users = pd.read_csv(RAW_DATA_DIR / "users_raw.csv")
products = pd.read_csv(RAW_DATA_DIR / "products_raw.csv")
orders = pd.read_csv(RAW_DATA_DIR / "orders_raw.csv")
order_items = pd.read_csv(RAW_DATA_DIR / "order_items_raw.csv")
reviews = pd.read_csv(RAW_DATA_DIR / "reviews_raw.csv")
events = pd.read_csv(RAW_DATA_DIR / "events_raw.csv")

## Create Working Copies

To preserve the integrity of the original raw datasets, all cleaning operations are performed on independent working copies.

In [ ]:
users_clean = users.copy()
products_clean = products.copy()
orders_clean = orders.copy()
order_items_clean = order_items.copy()
reviews_clean = reviews.copy()
events_clean = events.copy()

## Helper Functions

Reusable helper functions are created to standardize the cleaning workflow across all tables.

In [ ]:
def clean_text_column(df, column):
    """Strip extra whitespace and standardize text casing using title case."""
    df[column] = (
        df[column]
        .astype("string")
        .str.strip()
        .str.title()
    )
    return df


def remove_exact_duplicates(df):
    """Remove exact duplicate rows and return the cleaned dataframe plus row counts."""
    rows_before = len(df)
    df = df.drop_duplicates()
    rows_after = len(df)
    rows_removed = rows_before - rows_after

    return df, rows_before, rows_after, rows_removed


def keep_most_complete_record(df, key_column):
    """For duplicated keys, keep the record with the fewest missing values."""
    df = df.copy()
    df["_completeness_score"] = df.notna().sum(axis=1)
    df = df.sort_values(by="_completeness_score", ascending=False)
    df = df.drop_duplicates(subset=key_column, keep="first")
    df = df.drop(columns="_completeness_score")

    return df


def validation_summary(df, table_name, key_column=None):
    """Create a validation summary for a cleaned table."""
    summary = {
        "table": table_name,
        "rows": len(df),
        "columns": df.shape[1],
        "exact_duplicate_rows": df.duplicated().sum(),
        "missing_values_total": df.isna().sum().sum()
    }

    if key_column is not None:
        summary[f"duplicate_{key_column}"] = df[key_column].duplicated().sum()
        summary[f"missing_{key_column}"] = df[key_column].isna().sum()

    return pd.DataFrame([summary])

# 1. Users Cleaning

## Cleaning Strategy

The Users table should contain one unique record per customer. The cleaning process focuses on:

- Converting `signup_date` to datetime
- Converting invalid date values to `NaT`
- Removing exact duplicate rows
- Resolving duplicate `user_id` values by keeping the most complete customer record
- Standardizing categorical/text fields

## Convert `signup_date` to Datetime

In [ ]:
users_clean["signup_date"] = pd.to_datetime(
    users_clean["signup_date"],
    errors="coerce"
)

users_clean["signup_date"].dtype

## Remove Exact Duplicate Rows

In [ ]:
users_clean, users_rows_before, users_rows_after, users_rows_removed = remove_exact_duplicates(users_clean)

users_rows_before, users_rows_after, users_rows_removed

## Resolve Duplicate `user_id` Values

In [ ]:
users_clean = keep_most_complete_record(users_clean, "user_id")

users_clean["user_id"].duplicated().sum()

## Standardize Text Fields

In [ ]:
for col in ["gender", "city"]:
    users_clean = clean_text_column(users_clean, col)

users_clean[["gender", "city"]].head()

## Users Validation

In [ ]:
validation_summary(users_clean, "users", "user_id")

# 2. Products Cleaning

## Cleaning Strategy

The Products table should contain one unique record per product. The cleaning process focuses on:

- Removing exact duplicate rows
- Resolving duplicate `product_id` values by keeping the most complete product record
- Standardizing product categories
- Handling invalid prices
- Handling invalid product ratings

## Remove Exact Duplicate Rows

In [ ]:
products_clean, products_rows_before, products_rows_after, products_rows_removed = remove_exact_duplicates(products_clean)

products_rows_before, products_rows_after, products_rows_removed

## Resolve Duplicate `product_id` Values

In [ ]:
products_clean = keep_most_complete_record(products_clean, "product_id")

products_clean["product_id"].duplicated().sum()

## Standardize Product Categories

In [ ]:
products_clean = clean_text_column(products_clean, "category")

products_clean["category"].value_counts(dropna=False)

## Handle Invalid Prices

In [ ]:
# Negative or zero prices are not valid for product catalog analysis.
# They are converted to missing values and can be reviewed later if needed.

products_clean.loc[products_clean["price"] <= 0, "price"] = np.nan

(products_clean["price"] <= 0).sum()

## Handle Invalid Product Ratings

In [ ]:
# Ratings should be between 1 and 5.
# Invalid ratings are converted to missing values.

products_clean.loc[
    (products_clean["rating"] < 1) | (products_clean["rating"] > 5),
    "rating"
] = np.nan

((products_clean["rating"] < 1) | (products_clean["rating"] > 5)).sum()

## Products Validation

In [ ]:
validation_summary(products_clean, "products", "product_id")

# 3. Orders Cleaning

## Cleaning Strategy

The Orders table contains order-level transactions. The cleaning process focuses on:

- Converting `order_date` to datetime
- Standardizing `order_status`
- Removing exact duplicate rows
- Resolving duplicate `order_id` values
- Handling invalid negative order totals

## Convert `order_date` to Datetime

In [ ]:
orders_clean["order_date"] = pd.to_datetime(
    orders_clean["order_date"],
    errors="coerce"
)

orders_clean["order_date"].dtype

## Standardize `order_status`

In [ ]:
orders_clean = clean_text_column(orders_clean, "order_status")

orders_clean["order_status"].value_counts(dropna=False)

## Remove Exact Duplicate Rows

In [ ]:
orders_clean, orders_rows_before, orders_rows_after, orders_rows_removed = remove_exact_duplicates(orders_clean)

orders_rows_before, orders_rows_after, orders_rows_removed

## Resolve Duplicate `order_id` Values

In [ ]:
orders_clean = keep_most_complete_record(orders_clean, "order_id")

orders_clean["order_id"].duplicated().sum()

## Handle Invalid Order Totals

In [ ]:
# Negative order totals are not valid for revenue analysis.
# These values are converted to missing values for later review.

orders_clean.loc[orders_clean["total_amount"] < 0, "total_amount"] = np.nan

(orders_clean["total_amount"] < 0).sum()

## Orders Validation

In [ ]:
validation_summary(orders_clean, "orders", "order_id")

# 4. Order Items Cleaning

## Cleaning Strategy

The Order Items table contains item-level transaction details. The cleaning process focuses on:

- Removing exact duplicate rows
- Resolving duplicate `order_item_id` values
- Handling invalid quantities
- Handling invalid item prices

## Remove Exact Duplicate Rows

In [ ]:
order_items_clean, order_items_rows_before, order_items_rows_after, order_items_rows_removed = remove_exact_duplicates(order_items_clean)

order_items_rows_before, order_items_rows_after, order_items_rows_removed

## Resolve Duplicate `order_item_id` Values

In [ ]:
order_items_clean = keep_most_complete_record(order_items_clean, "order_item_id")

order_items_clean["order_item_id"].duplicated().sum()

## Handle Invalid Quantities

In [ ]:
# Quantity must be greater than zero for item-level sales analysis.
# Invalid quantities are converted to missing values.

order_items_clean.loc[order_items_clean["quantity"] <= 0, "quantity"] = np.nan

(order_items_clean["quantity"] <= 0).sum()

## Handle Invalid Item Prices

In [ ]:
# Item price must be greater than zero.
# Invalid item prices are converted to missing values.

order_items_clean.loc[order_items_clean["item_price"] <= 0, "item_price"] = np.nan

(order_items_clean["item_price"] <= 0).sum()

## Order Items Validation

In [ ]:
validation_summary(order_items_clean, "order_items", "order_item_id")

# 5. Reviews Cleaning

## Cleaning Strategy

The Reviews table contains customer product feedback. The cleaning process focuses on:

- Converting `review_date` to datetime
- Removing exact duplicate rows
- Resolving duplicate `review_id` values
- Handling invalid review ratings

## Convert `review_date` to Datetime

In [ ]:
reviews_clean["review_date"] = pd.to_datetime(
    reviews_clean["review_date"],
    errors="coerce"
)

reviews_clean["review_date"].dtype

## Remove Exact Duplicate Rows

In [ ]:
reviews_clean, reviews_rows_before, reviews_rows_after, reviews_rows_removed = remove_exact_duplicates(reviews_clean)

reviews_rows_before, reviews_rows_after, reviews_rows_removed

## Resolve Duplicate `review_id` Values

In [ ]:
reviews_clean = keep_most_complete_record(reviews_clean, "review_id")

reviews_clean["review_id"].duplicated().sum()

## Handle Invalid Review Ratings

In [ ]:
# Review ratings should be between 1 and 5.
# Invalid ratings are converted to missing values.

reviews_clean.loc[
    (reviews_clean["rating"] < 1) | (reviews_clean["rating"] > 5),
    "rating"
] = np.nan

((reviews_clean["rating"] < 1) | (reviews_clean["rating"] > 5)).sum()

## Reviews Validation

In [ ]:
validation_summary(reviews_clean, "reviews", "review_id")

# 6. Events Cleaning

## Cleaning Strategy

The Events table contains user behavioral events. The cleaning process focuses on:

- Converting `event_timestamp` to datetime
- Standardizing `event_type`
- Removing exact duplicate rows
- Resolving duplicate `event_id` values

## Convert `event_timestamp` to Datetime

In [ ]:
events_clean["event_timestamp"] = pd.to_datetime(
    events_clean["event_timestamp"],
    errors="coerce"
)

events_clean["event_timestamp"].dtype

## Standardize `event_type`

In [ ]:
events_clean = clean_text_column(events_clean, "event_type")

events_clean["event_type"].value_counts(dropna=False)

## Remove Exact Duplicate Rows

In [ ]:
events_clean, events_rows_before, events_rows_after, events_rows_removed = remove_exact_duplicates(events_clean)

events_rows_before, events_rows_after, events_rows_removed

## Resolve Duplicate `event_id` Values

In [ ]:
events_clean = keep_most_complete_record(events_clean, "event_id")

events_clean["event_id"].duplicated().sum()

## Events Validation

In [ ]:
validation_summary(events_clean, "events", "event_id")

# Final Validation

The following validation table summarizes the cleaned datasets after all cleaning operations have been completed.

In [ ]:
final_validation = pd.concat([
    validation_summary(users_clean, "users", "user_id"),
    validation_summary(products_clean, "products", "product_id"),
    validation_summary(orders_clean, "orders", "order_id"),
    validation_summary(order_items_clean, "order_items", "order_item_id"),
    validation_summary(reviews_clean, "reviews", "review_id"),
    validation_summary(events_clean, "events", "event_id")
], ignore_index=True)

final_validation

# Export Cleaned Datasets

The cleaned datasets are exported to the `data/cleaned` directory. These files will be used in the Data Integration, Exploratory Data Analysis, SQL, and Power BI stages of the project.

In [ ]:
users_clean.to_csv(CLEAN_DATA_DIR / "users_clean.csv", index=False)
products_clean.to_csv(CLEAN_DATA_DIR / "products_clean.csv", index=False)
orders_clean.to_csv(CLEAN_DATA_DIR / "orders_clean.csv", index=False)
order_items_clean.to_csv(CLEAN_DATA_DIR / "order_items_clean.csv", index=False)
reviews_clean.to_csv(CLEAN_DATA_DIR / "reviews_clean.csv", index=False)
events_clean.to_csv(CLEAN_DATA_DIR / "events_clean.csv", index=False)

list(CLEAN_DATA_DIR.iterdir())

# Executive Summary

## Objective

The objective of this notebook was to clean the raw e-commerce datasets and prepare them for integration and analysis. Six datasets were cleaned: Users, Products, Orders, Order Items, Reviews, and Events.

## Cleaning Operations Performed

The following cleaning operations were completed:

- Converted date and timestamp columns to proper datetime datatypes.
- Standardized invalid date strings as missing datetime values (`NaT`).
- Removed exact duplicate records across all tables.
- Resolved duplicate primary key values by retaining the most complete record.
- Standardized categorical and text fields where appropriate.
- Converted invalid numeric values such as negative prices, negative order amounts, invalid quantities, and ratings outside the valid range into missing values for review.
- Validated each cleaned table to confirm duplicate primary keys were resolved.
- Exported cleaned datasets for downstream analysis.

## Final Output

The cleaned datasets were exported to the `data/cleaned` directory and will be used in the next project stage.

## Conclusion

The cleaning process prioritized data integrity, transparency, reproducibility, and business relevance. Rather than applying generic cleaning techniques, each transformation was linked to a business rule and validated after implementation.

The cleaned datasets now provide a stronger foundation for data integration, exploratory analysis, SQL-based business analysis, Power BI dashboard development, and executive reporting.

## Next Steps

The next stage of the project is Data Integration. In that phase, the cleaned tables will be joined and validated to create an analysis-ready dataset suitable for business insights and dashboard development.